In [1]:
!pip install --upgrade --force-reinstall langchain>=0.1.17 openai>=1.13.3 langchain_openai>=0.1.6 transformers>=4.40.1 datasets>=2.18.0 accelerate>=0.27.2 sentence-transformers>=2.5.1 duckduckgo-search>=5.2.2 langchain_community
!CMAKE_ARGS="-DLLAMA_CUDA=on" pip install llama-cpp-python==0.2.69

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
dask-cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.12.5 which is incompatible.
cudf-cu12 25.10.0 requires pandas<2.4.0dev0,>=2.0, but you have pandas 3.0.1 which is inco

Loading an LLM

In [7]:
!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
from langchain_community.llms import LlamaCpp

# Make sure the model path is correct for your system!
llm = LlamaCpp(
    model_path="Phi-3-mini-4k-instruct-fp16.gguf",
    n_gpu_layers=-1,
    max_tokens=500,
    n_ctx=2048,
    seed=42,
    verbose=False
)

--2026-02-25 20:52:43--  https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-fp16.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.23, 18.164.174.55, 18.164.174.118, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.23|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/662698108f7573e6a6478546/a9cdcf6e9514941ea9e596583b3d3c44dd99359fb7dd57f322bb84a0adc12ad4?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27Phi-3-mini-4k-instruct-fp16.gguf%3B+filename%3D%22Phi-3-mini-4k-instruct-fp16.gguf%22%3B&Expires=1772056363&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiRXBvY2hUaW1lIjoxNzcyMDU2MzYzfX0sIlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjYyNjk4MTA4Zjc1NzNlNmE2NDc4NTQ2L2E5Y2RjZjZlOTUxNDk0MWVhOWU1OTY1ODNiM2QzYzQ0ZGQ5OTM1OWZiN2RkNTdmMzIyYmI4NGEwYWRjMTJhZDRcXD9yZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSoifV19

In [3]:
llm.invoke("Hi! My name is Maarten. What is 1 + 1?")

''

Chains

In [22]:
from langchain_core.prompts import PromptTemplate

# Create a prompt template with the "input_prompt" variable
template = """<s><|user|>
{input_prompt}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt"]
)

In [23]:
basic_chain = prompt | llm

In [24]:
basic_chain.invoke(
    {
        "input_prompt": "Hi! My name is Maarten. What is 1 + 1?",
    }
)

' Hello Maarten! The answer to 1 + 1 is 2.'

Multiple Chains

In [13]:
from langchain import LLMChain

# Create a chain for the title of our story
template = """<s><|user|>
Create a title for a story about {summary}. Only return the title.<|end|>
<|assistant|>"""
title_prompt = PromptTemplate(template=template, input_variables=["summary"])
title = LLMChain(llm=llm, prompt=title_prompt, output_key="title")

In [9]:
!pip show langchain
!ls -F $(python -c 'import langchain; import os; print(os.path.dirname(langchain.__file__))')/chains

Name: langchain
Version: 0.0.352
Summary: Building applications with LLMs through composability
Home-page: https://github.com/langchain-ai/langchain
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: aiohttp, dataclasses-json, jsonpatch, langchain-community, langchain-core, langsmith, numpy, pydantic, PyYAML, requests, SQLAlchemy, tenacity
Required-by: 
api/			   __init__.py		       openai_tools/
base.py			   llm_bash/		       prompt_selector.py
chat_vector_db/		   llm_checker/		       __pycache__/
combine_documents/	   llm_math/		       qa_generation/
constitutional_ai/	   llm.py		       qa_with_sources/
conversation/		   llm_requests.py	       query_constructor/
conversational_retrieval/  llm_summarization_checker/  question_answering/
elasticsearch_database/    llm_symbolic_math/	       retrieval_qa/
ernie_functions/	   loading.py		       router/
example_generator.py	   mapreduce.py		       sequential.py
flare/			   moderation.py	       

In [15]:
title.invoke({"summary": "a girl that lost her puppy"})

{'summary': 'a girl that lost her puppy',
 'title': ' "Whispers of Lily: The Journey to Find Her Lost Companion"'}

In [16]:
# Create a chain for the character description using the summary and title
template = """<s><|user|>
Describe the main character of a story about {summary} with the title {title}. Use only two sentences.<|end|>
<|assistant|>"""
character_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title"]
)
character = LLMChain(llm=llm, prompt=character_prompt, output_key="character")

In [17]:
# Create a chain for the story using the summary, title, and character description
template = """<s><|user|>
Create a story about {summary} with the title {title}. The main charachter is: {character}. Only return the story and it cannot be longer than one paragraph<|end|>
<|assistant|>"""
story_prompt = PromptTemplate(
    template=template, input_variables=["summary", "title", "character"]
)
story = LLMChain(llm=llm, prompt=story_prompt, output_key="story")

In [18]:
llm_chain = title | character | story

In [19]:
llm_chain.invoke("a girl that lost her puppy")

{'summary': 'a girl that lost her puppy',
 'title': ' "Whiskers\' Farewell: A Journey Through Heartache and Hope"',
 'character': " The protagonist, Lily, is an empathetic and resilient young girl who faces immense heartbreak after losing her beloved puppy Whiskers, but with the support of friends and family, she discovers strength within herself to navigate through this difficult time while finding hope for a brighter future. Throughout the story, Lily's determination to cherish memories of Whiskers and embrace new opportunities for growth and healing showcase her remarkable spirit and unwavering love for her furry friend.",
 'story': ' "Whiskers\' Farewell: A Journey Through Heartache and Hope" tells the poignant tale of Lily, an empathetic and resilient young girl whose world crumbles when she loses her beloved puppy Whiskers. Devastated by this profound loss, Lily spirals into a whirlwind of sorrow, but with unwavering support from friends and family, she begins to discover an inne

Memory

In [25]:
# Let's give the LLM our name
basic_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

' The answer to the question "What is 1 + 1?" is 2. This is a basic arithmetic operation, where you are adding one unit to another unit, resulting in two units altogether.'

In [26]:
# Next, we ask the LLM to reproduce the name
basic_chain.invoke({"input_prompt": "What is my name?"})

" I'm sorry, but as an AI, I don't have the ability to know personal information about individuals unless it has been shared with me in the course of our conversation. Therefore, I cannot determine your name."

ConversationBuffer

In [2]:
from langchain_core.prompts import PromptTemplate

# Create an updated prompt template to include a chat history
template = """<s><|user|>Current conversation:{chat_history}

{input_prompt}<|end|>
<|assistant|>"""

prompt = PromptTemplate(
    template=template,
    input_variables=["input_prompt", "chat_history"]
)

In [7]:
# Initialize an empty chat history
chat_history = []

def chat(user_input, llm):
    # Step 1: Add user input to history
    chat_history.append({"role": "user", "content": user_input})

    # Step 2: Construct prompt for LLM from full history
    prompt = ""
    for message in chat_history:
        prompt += f"{message['role'].capitalize()}: {message['content']}\n"

    # Step 3: Generate LLM response (pseudo-function)
    response = llm(prompt)  # replace with your actual LLM call

    # Step 4: Add assistant response to history
    chat_history.append({"role": "assistant", "content": response})

    return response

# Example usage
def fake_llm(prompt):
    # This is a stand-in for a real LLM call
    return f"[LLM response to: {prompt[:30]}...]"

print(chat("Hi, can you explain buffer memory?", fake_llm))
print(chat("How does it help long conversations?", fake_llm))

[LLM response to: User: Hi, can you explain buff...]
[LLM response to: User: Hi, can you explain buff...]


In [6]:
from langchain_community.memory import ConversationBufferMemory
from langchain_community.chains import LLMChain

# Define the type of Memory we will use
memory = ConversationBufferMemory(memory_key="chat_history")

# Chain the LLM, Prompt, and Memory together
llm_chain = LLMChain(
    prompt=prompt,
    llm=llm,
    memory=memory
)

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
# Generate a conversation and ask a basic question
llm_chain.invoke({"input_prompt": "Hi! My name is Maarten. What is 1 + 1?"})

In [ ]:
# Does the LLM remember the name we gave it?
llm_chain.invoke({"input_prompt": "What is my name?"})